# UE Data Mining - Projet

**Nom du groupe :** à compléter  
**Membres du groupe :** à compléter

Ce notebook constitue le rapport d'exploration des documents textuels du projet. Toutes les étapes peuvent être rejouées après avoir changé le dossier analysé.

## Objectifs et méthode

Les quatre sources textuelles étudiées sont :

- les documents publics et d'investigation ;
- les mémos institutionnels ;
- les notes cliniques ;
- les rapports de terrain environnementaux.

Le module document_analysis.py harmonise les colonnes des fichiers CSV, nettoie les textes, calcule la longueur des documents et fournit les fonctions de tri, filtrage, recherche, fréquences de mots et corrélation de Spearman. Les données sont fictives et servent de corpus d'expérimentation.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from document_analysis import (
    category_counts,
    document_summary,
    filter_documents,
    get_analysis_folder,
    load_active_documents,
    ngram_frequencies,
    output_directory,
    quality_report,
    save_output,
    search_documents,
    set_analysis_folder,
    sort_documents,
    spearman_correlation,
    spearman_correlation_by_group,
    word_frequencies,
    word_frequencies_by_group,
)

print(f'Matplotlib version : {matplotlib.__version__}')
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 80)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Sélection du corpus

La variable ci-dessous est le switcher global du notebook. Valeurs possibles :

- all : les quatre sources textuelles ;
- public_records : documents publics/d'investigation et mémos institutionnels ;
- health : notes cliniques ;
- ecology : rapports de terrain environnementaux.

Pour analyser un autre dossier, modifier uniquement ANALYSIS_FOLDER, puis exécuter à nouveau les cellules dans l'ordre.

In [ ]:
ANALYSIS_FOLDER = 'all'

set_analysis_folder(ANALYSIS_FOLDER)
documents = load_active_documents()

print(f'Dossier analysé : {get_analysis_folder()}')
print(f'Nombre de documents : {len(documents)}')
print(f'Nombre de colonnes : {len(documents.columns)}')
print(f'Les résultats seront enregistrés dans : {output_directory()}')
display(documents.head(3))

### Colonnes ajoutées pour l'analyse

En plus des colonnes originales, le chargement crée notamment document_title, document_text, text_clean, search_text, word_count, character_count, document_year, document_month, source_id et source_family. Ces noms communs permettent de comparer les quatre sources malgré leurs schémas différents.

In [ ]:
summary_by_source = document_summary(documents, group_by='source_id')
summary_by_type = document_summary(documents, group_by='document_type')

print('Résumé par source')
display(summary_by_source)

print('Résumé par type de document')
display(summary_by_type.head(15))

## 2. Qualité et préparation des données

On vérifie les identifiants, les titres, les dates et les textes vides. Les textes sont normalisés par le module afin de supprimer les retours à la ligne et les espaces répétés, tout en conservant les colonnes originales.

In [ ]:
quality = quality_report(documents)
display(quality)

missing_values = (
    documents.isna()
    .sum()
    .rename('missing_count')
    .to_frame()
    .query('missing_count > 0')
    .sort_values('missing_count', ascending=False)
)

print('Colonnes contenant des valeurs manquantes')
display(missing_values if not missing_values.empty else pd.DataFrame({'message': ['Aucune valeur manquante']}))

duplicate_ids = documents[documents['document_id'].duplicated(keep=False)].sort_values('document_id')
print(f'Identifiants dupliqués : {len(duplicate_ids)} ligne(s)')
display(duplicate_ids[['document_id', 'source_id', 'document_title']].head(20))

## 3. Répartition des documents

Les effectifs sont calculés par source, type de document et statut d'accès. Ces tableaux donnent une première vision de la composition du corpus.

In [ ]:
source_counts = category_counts(documents, 'source_id')
type_counts = category_counts(documents, 'document_type')
access_counts = category_counts(documents, 'access_status')

display(source_counts)
display(type_counts)
display(access_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

source_counts.sort_values('document_count').plot.barh(
    x='source_id', y='document_count', ax=axes[0], legend=False, color='#4472C4'
)
axes[0].set_title('Nombre de documents par source')
axes[0].set_xlabel('Nombre de documents')
axes[0].set_ylabel('Source')

type_counts.head(10).sort_values('document_count').plot.barh(
    x='document_type', y='document_count', ax=axes[1], legend=False, color='#70AD47'
)
axes[1].set_title('Top 10 des types de documents')
axes[1].set_xlabel('Nombre de documents')
axes[1].set_ylabel('Type')

plt.tight_layout()
plt.show()

## 4. Analyse temporelle

La date est convertie en date pandas lors du chargement. On observe ici la période couverte par le corpus et le nombre de documents par année.

In [ ]:
dated_documents = documents.dropna(subset=['document_date']).copy()
year_counts = (
    dated_documents.groupby('document_year')
    .size()
    .rename('document_count')
    .reset_index()
)

print(f"Date la plus ancienne : {dated_documents['document_date'].min().date()}")
print(f"Date la plus récente : {dated_documents['document_date'].max().date()}")
display(year_counts)

plt.figure(figsize=(14, 5))
plt.plot(year_counts['document_year'], year_counts['document_count'], marker='o')
plt.title('Nombre de documents par année')
plt.xlabel('Année')
plt.ylabel('Nombre de documents')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Longueur et contenu textuel

word_count mesure le nombre de mots du texte normalisé et character_count le nombre de caractères. Les statistiques et les graphiques permettent de repérer les documents atypiquement courts ou longs.

In [ ]:
length_statistics = documents[['word_count', 'character_count']].describe().T
display(length_statistics)

longest_documents = sort_documents(
    documents, by='word_count', ascending=False
)[
    ['document_id', 'source_id', 'document_type', 'document_date', 'document_title', 'word_count', 'character_count']
].head(10)
display(longest_documents)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(documents['word_count'], bins=15, color='#ED7D31', edgecolor='white')
axes[0].set_title('Distribution du nombre de mots')
axes[0].set_xlabel('Nombre de mots')
axes[0].set_ylabel('Nombre de documents')

axes[1].scatter(documents['word_count'], documents['character_count'], alpha=0.7, color='#A5A5A5')
axes[1].set_title('Mots et caractères')
axes[1].set_xlabel('Nombre de mots')
axes[1].set_ylabel('Nombre de caractères')

plt.tight_layout()
plt.show()

## 6. Fréquences de mots et expressions

Les mots vides anglais sont retirés afin de faire ressortir le vocabulaire thématique. La fréquence d'un terme compte ses occurrences, tandis que sa fréquence documentaire compte le nombre de documents dans lesquels il apparaît.

In [ ]:
frequencies = word_frequencies(documents, min_frequency=2)
top_words = frequencies.head(20)
display(top_words)

if not top_words.empty:
    plt.figure(figsize=(10, 7))
    top_words.sort_values('term_frequency').plot.barh(
        x='term', y='term_frequency', legend=False, ax=plt.gca(), color='#5B9BD5'
    )
    plt.title('Top 20 des mots les plus fréquents')
    plt.xlabel('Fréquence totale')
    plt.ylabel('Mot')
    plt.tight_layout()
    plt.show()

In [ ]:
frequencies_by_source = word_frequencies_by_group(
    documents,
    group_by='source_id',
    min_frequency=2,
)

print('Vocabulaire fréquent par source')
display(frequencies_by_source.groupby('source_id', group_keys=False).head(10))

bigrams = ngram_frequencies(documents, n=2, min_frequency=2)
trigrams = ngram_frequencies(documents, n=3, min_frequency=2)

print('Bigrams les plus fréquents')
display(bigrams.head(15))
print('Trigrams les plus fréquents')
display(trigrams.head(15))

## 7. Recherche, tri et filtrage

La recherche porte sur search_text, qui regroupe le titre et le texte nettoyé. Les fonctions de filtrage acceptent aussi le type, la source, le statut d'accès et une période de dates.

In [ ]:
QUERY = 'shipment'
search_results = search_documents(documents, QUERY)
print(f"Résultats pour la recherche {QUERY!r} : {len(search_results)}")
display(search_results[
    ['document_id', 'source_id', 'document_date', 'document_type', 'document_title']
].head(10))

restricted_documents = filter_documents(documents, access_status='restricted')
print(f'Documents restreints : {len(restricted_documents)}')
display(restricted_documents[
    ['document_id', 'source_id', 'document_date', 'document_title', 'access_status']
].head(10))

recent_documents = filter_documents(documents, after='2020-01-01')
print(f'Documents depuis 2020 : {len(recent_documents)}')
display(sort_documents(recent_documents, by='document_date', ascending=False)[
    ['document_id', 'source_id', 'document_date', 'document_title']
].head(10))

## 8. Corrélations de Spearman

La corrélation de Spearman mesure une association monotone entre deux variables numériques à partir de leurs rangs. Elle est adaptée ici pour comparer la longueur des documents et leur année, sans supposer une relation linéaire stricte.

Une corrélation proche de 1 ou de -1 indique une association forte ; une valeur proche de 0 indique une association faible. La diagonale vaut 1 par construction et ne doit pas être interprétée comme une relation entre deux variables différentes.

In [ ]:
CORRELATION_COLUMNS = ['word_count', 'character_count', 'document_year']
available_correlation_columns = [
    column for column in CORRELATION_COLUMNS
    if column in documents.columns and documents[column].notna().any()
]

if len(available_correlation_columns) >= 2:
    correlation = spearman_correlation(
        documents,
        columns=available_correlation_columns,
    )
    display(correlation)

    plt.figure(figsize=(7, 5))
    plt.imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(label='Corrélation de Spearman')
    plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=45, ha='right')
    plt.yticks(range(len(correlation.index)), correlation.index)
    for row in range(len(correlation.index)):
        for column in range(len(correlation.columns)):
            value = correlation.iloc[row, column]
            if pd.notna(value):
                plt.text(column, row, f'{value:.2f}', ha='center', va='center')
    plt.title('Matrice de corrélation de Spearman')
    plt.tight_layout()
    plt.show()
else:
    correlation = pd.DataFrame()
    print('Pas assez de variables numériques disponibles pour calculer une corrélation.')

In [ ]:
if len(available_correlation_columns) >= 2:
    correlations_by_source = spearman_correlation_by_group(
        documents,
        group_by='source_id',
        columns=available_correlation_columns,
    )
    for source_id, matrix in correlations_by_source.items():
        print(f'Corrélations pour la source : {source_id}')
        display(matrix)
else:
    correlations_by_source = {}


## 9. Interprétation des résultats

Pour rédiger l'analyse, commenter les points suivants en s'appuyant sur les tableaux et graphiques précédents :

1. La composition du corpus : quelles sources et quels types sont majoritaires ?
2. La qualité : y a-t-il des identifiants, dates, titres ou textes manquants ?
3. La période couverte : les documents sont-ils concentrés sur certaines années ?
4. La longueur : quelles sources produisent les documents les plus longs ? Existe-t-il des valeurs atypiques ?
5. Le vocabulaire : quels termes et groupes de mots caractérisent le corpus ?
6. Les relations numériques : la longueur en mots est-elle liée à la longueur en caractères ? L'année est-elle associée à la longueur des documents ?

Attention : une corrélation décrit une association dans ce corpus ; elle ne suffit pas à démontrer un lien de causalité.

In [ ]:
print('Résumé automatique du corpus')
print(f"- {len(documents)} document(s) dans le dossier {get_analysis_folder()!r}")
print(f"- {documents['source_id'].nunique()} source(s) de données")
print(f"- {documents['document_type'].nunique()} type(s) de documents")
print(f"- Longueur médiane : {documents['word_count'].median():.0f} mots")
print(f"- Statut d'accès majoritaire : {documents['access_status'].value_counts().idxmax()}")

if not top_words.empty:
    print(f"- Terme le plus fréquent après nettoyage : {top_words.iloc[0]['term']!r}")
if not correlation.empty:
    distinct_pairs = correlation.copy()
    for column in distinct_pairs.columns:
        distinct_pairs.loc[column, column] = pd.NA
    strongest_pair = distinct_pairs.abs().stack().dropna()
    if not strongest_pair.empty:
        pair = strongest_pair.idxmax()
        print(f"- Association la plus forte : {pair[0]} / {pair[1]} ({correlation.loc[pair[0], pair[1]]:.3f})")

## 10. Export des résultats

Les fichiers sont enregistrés automatiquement dans outputs/<dossier_analyse>/. Le chemin change donc avec ANALYSIS_FOLDER, ce qui évite de mélanger les résultats des différents corpus.

In [ ]:
exported_files = {}
exported_files['summary_by_source'] = save_output(summary_by_source, 'summary_by_source.csv')
exported_files['summary_by_type'] = save_output(summary_by_type, 'summary_by_type.csv')
exported_files['quality_report'] = save_output(quality, 'quality_report.csv')
exported_files['source_counts'] = save_output(source_counts, 'source_counts.csv')
exported_files['type_counts'] = save_output(type_counts, 'type_counts.csv')
exported_files['access_counts'] = save_output(access_counts, 'access_counts.csv')
exported_files['word_frequencies'] = save_output(frequencies, 'word_frequencies.csv')
exported_files['bigrams'] = save_output(bigrams, 'bigrams.csv')
exported_files['trigrams'] = save_output(trigrams, 'trigrams.csv')

if not correlation.empty:
    exported_files['spearman_correlation'] = save_output(
        correlation,
        'spearman_correlation.csv',
        index=True,
    )

for result_name, path in exported_files.items():
    print(f'{result_name}: {path}')

## 11. Comparaison rapide des dossiers

Cette dernière cellule montre les effectifs disponibles dans chaque périmètre. Elle n'écrase pas le choix global effectué au début du notebook.

In [ ]:
folder_comparison = pd.DataFrame(
    [
        {
            'analysis_folder': folder,
            'document_count': len(load_active_documents(folder=folder)),
        }
        for folder in ['all', 'public_records', 'health', 'ecology']
    ]
)
display(folder_comparison)

print('Pour changer le corpus : modifier ANALYSIS_FOLDER dans la cellule de sélection, puis utiliser Exécuter tout.')